In [13]:
"""
Excluding Experimental+SS literature
SVM (SVC) Training + Prediction Script
Python 3.10
Dependencies: pandas, numpy, scikit-learn, joblib
"""

import pandas as pd
import numpy as np
import joblib
import warnings
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

# -------------------------------
# 1. Load dataset
# -------------------------------
dataset_path = r"C:\Users\gurra\Downloads\Excluding_30_ss316l.csv"
dataset = pd.read_csv(dataset_path)

# -------------------------------
# 2. Features & Target
# -------------------------------
features = [
    'power','speed','thickness','dia','Solidt','Sdensity','Sspheat','Sthercondu',
    'Liquidt','Ldensity','LSpheat','Lthercondu','Lsurfacet','Lviscosity',
    'Lfusion','dsigma','Absorptivity','Lvapor'
]
X = dataset[features]
y = dataset['defect']

# -------------------------------
# 3. Scale features
# -------------------------------
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split (here using full dataset for training if desired)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.1, random_state=42)

# -------------------------------
# 4. SVM + GridSearchCV
# -------------------------------
param_grid = {
    "C": [0.1, 1, 10, 50, 100, 150, 180, 200, 250],
    "kernel": ["linear", "rbf", "poly", "sigmoid"],
    "gamma": ["scale", "auto"]
}

svm_model = SVC(probability=True, random_state=42)

grid = GridSearchCV(
    estimator=svm_model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print("\nBest Parameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

# -------------------------------
# 5. Evaluate model
# -------------------------------
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

print("\nTrain Accuracy:", accuracy_score(y_train, y_train_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nConfusion Matrix (Test Set):\n", confusion_matrix(y_test, y_test_pred))
print("\nClassification Report (Test Set):\n", classification_report(y_test, y_test_pred, labels=[0,1,2,3]))

# -------------------------------
# 6. Save model & scaler
# -------------------------------
joblib.dump(best_model, "SVC_model.pkl")
joblib.dump(scaler, "SVC_scaler.pkl")

# -------------------------------
# 7. Predict new alloy dataset
# -------------------------------
new_data_path = r"C:\Users\gurra\Downloads\final 5-800w power speed ss316l alloy predictions (1).csv"
new_data = pd.read_csv(new_data_path)

# Scale features
X_new_scaled = scaler.transform(new_data[features])

# Predictions & probabilities
y_new_pred = best_model.predict(X_new_scaled)
y_new_proba = best_model.predict_proba(X_new_scaled)

# Map numeric predictions to labels
class_map = {0: 'Good', 1: 'Balling', 2: 'Lack of fusion', 3: 'Keyholing'}
new_data['Predicted_class'] = y_new_pred
new_data['Predicted_Label'] = new_data['Predicted_class'].map(class_map)

# Add probability columns
proba_df = pd.DataFrame(
    y_new_proba,
    columns=[f"Prob_{class_map[c]}" for c in sorted(class_map.keys())]
)
new_data = pd.concat([new_data, proba_df], axis=1)

# Save predictions
output_file = r"C:\Users\gurra\Downloads\EE_SS_SVC_predictions.csv"
new_data.to_csv(output_file, index=False)
print(f"\nPredictions saved to: {output_file}")
joblib.dump(best_model, r"C:\Users\gurra\Downloads\SVC_model.pkl")
joblib.dump(scaler, r"C:\Users\gurra\Downloads\SVC_scaler.pkl")


Fitting 5 folds for each of 72 candidates, totalling 360 fits

Best Parameters: {'C': 180, 'gamma': 'auto', 'kernel': 'rbf'}
Best CV Accuracy: 0.761904761904762

Train Accuracy: 0.9015873015873016
Test Accuracy: 0.8333333333333334

Confusion Matrix (Test Set):
 [[14  1  0  2]
 [ 2  7  1  0]
 [ 0  0  3  0]
 [ 0  0  0  6]]

Classification Report (Test Set):
               precision    recall  f1-score   support

           0       0.88      0.82      0.85        17
           1       0.88      0.70      0.78        10
           2       0.75      1.00      0.86         3
           3       0.75      1.00      0.86         6

    accuracy                           0.83        36
   macro avg       0.81      0.88      0.84        36
weighted avg       0.84      0.83      0.83        36


Predictions saved to: C:\Users\gurra\Downloads\EE_SS_SVC_predictions.csv


['C:\\Users\\gurra\\Downloads\\SVC_scaler.pkl']

In [14]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import os

# -------------------------------
# 1. Load new alloy dataset
# -------------------------------
new_data_path = r"C:\Users\gurra\Downloads\final 5-800w power speed ss316l alloy predictions (1).csv"
new_data = pd.read_csv(new_data_path)

features = [
    'power', 'speed', 'thickness', 'dia', 'Solidt', 'Sdensity', 'Sspheat',
    'Sthercondu', 'Liquidt', 'Ldensity', 'LSpheat', 'Lthercondu',
    'Lsurfacet', 'Lviscosity', 'Lfusion', 'dsigma', 'Absorptivity', 'Lvapor'
]

# -------------------------------
# 2. Load trained SVC model & scaler
# -------------------------------
model = joblib.load(r"C:\Users\gurra\Downloads\SVC_model.pkl")
scaler = joblib.load(r"C:\Users\gurra\Downloads\SVC_scaler.pkl")

# Scale new data
X_new_scaled = scaler.transform(new_data[features])

# -------------------------------
# 3. Predictions and probabilities
# -------------------------------
y_pred = model.predict(X_new_scaled)
y_proba = model.predict_proba(X_new_scaled)

class_map = {0:'Good', 1:'Balling', 2:'Lack of fusion', 3:'Keyholing'}
new_data['Predicted_class'] = y_pred
new_data['Predicted_Label'] = new_data['Predicted_class'].map(class_map)

# Add probabilities
proba_df = pd.DataFrame(
    y_proba,
    columns=[f"Prob_{class_map[c]}" for c in sorted(class_map.keys())]
)
new_data = pd.concat([new_data, proba_df], axis=1)

# Save predictions
output_file = r"c:\users\gurra\downloads\SVC_predictions.csv"
if os.path.exists(output_file):
    os.remove(output_file)
new_data.to_csv(output_file, index=False)
print(f"\n✅ Predictions saved to: {output_file}")

# -------------------------------
# 4. SHAP Analysis (KernelExplainer for SVC)
# -------------------------------
# Use small background set to speed up KernelExplainer
background = X_new_scaled[np.random.choice(X_new_scaled.shape[0], min(50, X_new_scaled.shape[0]), replace=False)]
explainer = shap.KernelExplainer(model.predict_proba, background)

# Compute SHAP values (list of arrays: one per class)
shap_values = explainer.shap_values(X_new_scaled, nsamples=100)

# -------------------------------
# 5. Beeswarm plot per class
# -------------------------------
for c, class_name in enumerate(class_map.values()):
    print(f"\nGenerating beeswarm for {class_name} ...")
    plt.figure(figsize=(10,6))
    shap.summary_plot(shap_values[c], X_new_scaled, feature_names=features, show=False)
    plt.title(f"SHAP Beeswarm – {class_name}", fontsize=16, fontweight='bold')
    plt.xlabel("SHAP values", fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()
    plt.show()

# -------------------------------
# 6. Stacked horizontal bar chart (mean |SHAP| per class)
# -------------------------------
mean_abs_per_class = np.array([np.mean(np.abs(sv), axis=0) for sv in shap_values])  # shape: n_classes x n_features
total_importance = np.sum(mean_abs_per_class, axis=0)
sorted_idx = np.argsort(total_importance)[::-1]
features_sorted = np.array(features)[sorted_idx]
mean_abs_sorted = mean_abs_per_class[:, sorted_idx]

colors = ["#08306b", "#377eb8", "#fbb4ae", "#e41a1c"]
class_names = [class_map[i] for i in range(len(class_map))]

bottom = np.zeros(len(features_sorted))
plt.figure(figsize=(10,6))
for i, (label, color) in enumerate(zip(class_names, colors)):
    plt.barh(features_sorted, mean_abs_sorted[i], left=bottom, color=color, label=label)
    bottom += mean_abs_sorted[i]

plt.xlabel("Mean |SHAP value|", fontsize=14)
plt.title("Stacked Feature Importance Across Classes", fontsize=16, fontweight='bold')
plt.legend(fontsize=12, title_fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()



✅ Predictions saved to: c:\users\gurra\downloads\SVC_predictions.csv


  0%|          | 0/48000 [00:00<?, ?it/s]


Generating beeswarm for Good ...


AssertionError: The shape of the shap_values matrix does not match the shape of the provided data matrix.

<Figure size 1000x600 with 0 Axes>